In [1]:
import K_Means_Clustering as km
import os
import pandas as pd
import trade_pairs as tp
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


In [2]:
year = 2015
for i in range(11):
    km.get_cointegrated_stocks_by_year(year+i)

has NaN values: False
Optimum Clusters: 6
has NaN values: False
Optimum Clusters: 6
has NaN values: False
Optimum Clusters: 7
has NaN values: False
Optimum Clusters: 7
has NaN values: False
Optimum Clusters: 7
has NaN values: False
Optimum Clusters: 8
has NaN values: False
Optimum Clusters: 7
has NaN values: False
Optimum Clusters: 7
has NaN values: False
Optimum Clusters: 6
has NaN values: False
Optimum Clusters: 9
has NaN values: False
Optimum Clusters: 8


In [3]:
year = 2015
trades = []
cointegrated_pairs_path = f"data/cointegrated-pairs-{year}.csv"
# if not os.path.exists(cointegrated_pairs_path):
#     df_cointegrated_pairs = km.get_cointegrated_stocks_by_year(2026 + i)
# else:
#     df_cointegrated_pairs = pd.read_csv(cointegrated_pairs_path)

df_returns = km.get_stock_returns_upto_year(year + 1)


In [4]:
temp = df_returns[df_returns["date"] >= 20140000]
temp
print("Does '14322' exist?", 'crsp_23536' in df_returns['id'].values)

Does '14322' exist? True


In [11]:
df_returns.tail(100000)

,date,id,stock_ret,price_index
697385,20140630,crsp_88912,-0.024261,100.0
697386,20140630,crsp_88924,0.016787,100.0
697387,20140630,crsp_88926,0.119318,100.0
697388,20140630,crsp_88940,0.018406,100.0
697389,20140630,crsp_88944,-0.018493,100.0
...,...,...,...,...
797380,20151231,crsp_93428,-0.116662,100.0
797381,20151231,crsp_93429,-0.098047,100.0
797382,20151231,crsp_93433,-0.264706,100.0
797383,20151231,crsp_93434,-0.040909,100.0


In [2]:
df_combined = pd.DataFrame()
year = 2015
df_returns = km.get_stock_returns_upto_year(2026)
for i in range(11):
    cointegrated_pairs_path = f"data/cointegrated-pairs-{year+i}.csv"
    if not os.path.exists(cointegrated_pairs_path):
        df_cointegrated_pairs = km.get_cointegrated_stocks_by_year(year)
    else:
        df_cointegrated_pairs = pd.read_csv(cointegrated_pairs_path)
    
    for j in range(len(df_cointegrated_pairs)):
        stock1_label = df_cointegrated_pairs.iloc[j, 0]
        stock2_label = df_cointegrated_pairs.iloc[j, 1]
        series1 = df_returns[df_returns["id"] == stock1_label].copy()
        series2 = df_returns[df_returns["id"] == stock2_label].copy()
        series1["price_index"] = (1 + series1["stock_ret"]).cumprod() * 1
        series2["price_index"] = (1 + series2["stock_ret"]).cumprod() * 1
    
        len(series1["stock_ret"].dropna())
        # Find the first index where date >= 20150000
        first_idx = series1[series1["date"] >= (year+i) * 10000].index[0]
        
        # Get position of that index
        pos = series1.index.get_loc(first_idx)
        
        # Get 6 rows before that position, plus alldone rows from that position onward
        series1 = series1.iloc[max(0, pos-6):]
        
        series1 = series1[series1["date"] < (year+i+1)*10000]



        
        first_idx = series2[series2["date"] >= (year+i)*10000].index[0]
        
        # Get position of that index
        pos = series2.index.get_loc(first_idx)
        
        # Get 6 rows before that position, plus all rows from that position onward
        series2 = series2.iloc[max(0, pos-6):]
        series2 = series2[series2["date"] < (year+i+1)*10000]
        try:
            temp, temp2 = tp.pairs_trade_monthly_with_risk(series1.set_index('date')["price_index"], series2.set_index('date')["price_index"],
                                                stock1_label, stock2_label)
            df_combined = pd.concat([df_combined, temp2], ignore_index=True)
        except:
            print("skip pair")
    print(year+i, "done")


2015 done
2016 done
2017 done
2018 done
2019 done
2020 done
2021 done
2022 done
2023 done
2024 done
2025 done


In [15]:
df_combined = df_combined[df_combined["holding_months"] > 0]

print(df_combined["pnl_dollars"].sum())
df_combined
df_combined

851.249119343938


,entry_date,exit_date,direction,entry_spread,exit_spread,pnl_dollars,holding_months,close_reason
0,20150130,20150331,SHORT,2.333910,0.911094,11.037340,2,z_cross
1,20150130,20150227,SHORT,-1.370083,-0.518895,1.262078,1,z_cross
2,20150930,20151231,LONG,-0.722755,-1.032880,-6.376105,3,stop_loss
3,20150130,20150331,LONG,-0.594720,-0.727829,3.068234,2,z_cross
4,20150430,20151030,LONG,-0.888278,0.639903,15.147256,6,max_holding
5,20151130,20151231,SHORT,0.836750,0.999899,1.153726,1,forced_liquidation_end_of_sample
6,20150227,20150331,SHORT,2.263286,1.396075,2.770283,1,z_cross
7,20150731,20150831,LONG,1.724290,1.875702,-4.128106,1,stop_loss
8,20150930,20151030,LONG,1.614869,2.866893,14.319756,1,z_cross
9,20150130,20150227,LONG,5.149402,4.451687,-0.133191,1,stop_loss


In [23]:
# Convert to datetime
df_combined["exit_date"] = pd.to_datetime(df_combined["exit_date"], format="%Y%m%d")

# Extract year-month period
df_combined["year_month"] = df_combined["exit_date"].dt.to_period("M")

# Count trades per month
count_per_month = df_combined.groupby("year_month").size().reset_index(name="count")

# Create full monthly range
full_range = pd.period_range(start=pd.to_datetime(20150130, format="%Y%m%d"), end=df_combined["exit_date"].max(), freq="M")
full_df = pd.DataFrame({"year_month": full_range})

# Merge and fill missing with 0
count_per_month_full = full_df.merge(count_per_month, on="year_month", how="left").fillna(0)
count_per_month_full["count"] = count_per_month_full["count"].astype(int)
count_per_month_full.to_csv("pairs-trade-counts-per-month.csv")

In [5]:
new_df = (
    df_combined.groupby("exit_date")["pnl_dollars"]
      .mean()  # average PnL across rows with same exit_date
      .reset_index(name="avg_pnl_dollars")
)
# new_df["avg_pnl_dollars"] = new_df["avg_pnl_dollars"].cumsum()
new_df

,exit_date,avg_pnl_dollars
0,20150227,2.241808
1,20150331,0.760594
2,20150430,4.615989
3,20150529,2.853412
4,20150630,1.857195
...,...,...
110,20250228,3.495048
111,20250331,-1.066412
112,20250430,-1.170146
113,20250530,-6.109636


In [6]:
import numpy as np
new_df["avg_pnl_dollars"] = new_df["avg_pnl_dollars"] / 100 + 1
new_df["log"] = np.log(new_df["avg_pnl_dollars"])
new_df["sum"] = new_df["log"].cumsum()
new_df["bench"] = np.exp(new_df["sum"]) - 1
pd.set_option('display.max_rows', None)
new_df

,exit_date,avg_pnl_dollars,log,sum,bench
0,20150227,1.022418,0.022170,0.022170,0.022418
1,20150331,1.007606,0.007577,0.029748,0.030195
2,20150430,1.046160,0.045126,0.074874,0.077748
3,20150529,1.028534,0.028135,0.103008,0.108501
4,20150630,1.018572,0.018402,0.121410,0.129088
5,20150731,1.072956,0.070418,0.191828,0.211462
6,20150831,0.960013,-0.040809,0.151019,0.163019
7,20150930,0.975845,-0.024451,0.126568,0.134926
8,20151030,1.069813,0.067484,0.194052,0.214160
9,20151130,0.962607,-0.038110,0.155942,0.168759


In [11]:
new_df.to_csv("pair-trade-per-month.csv")

In [9]:
df = pd.read_csv("data/mkt_ind.csv")
df = df[df["year"] >= 2015]
df["ret"] = df["ret"] + 1
df["log"] = np.log(df["ret"])
df["cumulative log"] = df["log"].cumsum()
df["bench"] = np.exp(df["cumulative log"]) - 1
df

,rf,year,month,ret,log,cumulative log,bench
120,0.0000,2015,1,0.968959,-0.031533,-0.031533,-0.031041
121,0.0000,2015,2,1.054893,0.053439,0.021906,0.022148
122,0.0000,2015,3,0.982604,-0.017549,0.004357,0.004366
123,0.0000,2015,4,1.008521,0.008485,0.012842,0.012924
124,0.0000,2015,5,1.010491,0.010437,0.023278,0.023551
125,0.0000,2015,6,0.978988,-0.021236,0.002043,0.002045
126,0.0000,2015,7,1.019742,0.019550,0.021592,0.021827
127,0.0000,2015,8,0.937419,-0.064625,-0.043032,-0.042120
128,0.0000,2015,9,0.973557,-0.026799,-0.069831,-0.067449
129,0.0000,2015,10,1.082983,0.079719,0.009888,0.009937


In [10]:
df.to_csv("pair-trade-per-month.csv")

In [4]:

df_cum_returns = df_combined[df_combined["holding_months"] > 0]
df_cum_returns = df_cum_returns.sort_values(by="exit_date")
df_cum_returns["cumulative_pnl"] = df_cum_returns["pnl_dollars"].cumsum()
df_final = df_cum_returns[["exit_date", "cumulative_pnl"]].drop_duplicates(subset="exit_date", keep="last")

In [23]:
df = pd.read_csv("data/mkt_ind.csv")
df["ret"].cumsum()

0     -0.025290
1     -0.006387
2     -0.025505
3     -0.045613
4     -0.015661
         ...   
241    1.824657
242    1.767113
243    1.759488
244    1.821012
245    1.870618
Name: ret, Length: 246, dtype: float64

In [ ]:
len(series1["stock_ret"].dropna())
# Find the first index where date >= 20150000
first_idx = series1[series1["date"] >= 20150000].index[0]

# Get position of that index
pos = series1.index.get_loc(first_idx)

# Get 6 rows before that position, plus all rows from that position onward
series1 = series1.iloc[max(0, pos-6):]

series1 = series1[series1["date"] < 20160000]

first_idx = series2[series2["date"] >= 20150000].index[0]

# Get position of that index
pos = series2.index.get_loc(first_idx)

# Get 6 rows before that position, plus all rows from that position onward
series2 = series2.iloc[max(0, pos-6):]
series2 = series2[series2["date"] < 20160000]

df = pd.DataFrame({
    'p1': series1.set_index('date')["stock_ret"], 
    'p2': series2.set_index('date')["stock_ret"]
}).dropna().copy()


In [ ]:
df

In [61]:
foo = tp.pairs_trade_monthly_with_risk(series1.set_index('date')["stock_ret"], series2.set_index('date')["stock_ret"])
foo

const   -0.006543
p2       1.400523
dtype: float64
const   -0.006403
p2       1.401835
dtype: float64
const   -0.004313
p2       1.363977
dtype: float64
const   -0.006588
p2       1.377345
dtype: float64
const   -0.002709
p2       1.330490
dtype: float64
const   -0.003752
p2       1.332470
dtype: float64
const   -0.003741
p2       1.332244
dtype: float64
const   -0.004297
p2       1.312564
dtype: float64
const   -0.002955
p2       1.295675
dtype: float64
const   -0.001791
p2       1.324493
dtype: float64
const   -0.003005
p2       1.326607
dtype: float64
const   -0.005005
p2       1.334433
dtype: float64


,entry_date,exit_date,direction,entry_spread,exit_spread,pnl_dollars,holding_months,close_reason
0,20150430,20150630,SHORT,0.027934,-0.003623,24250.418601,2,z_cross
1,20151130,20151231,LONG,-0.035688,0.019019,35709.582002,1,forced_liquidation_end_of_sample


In [7]:
series1 = df_returns[stock1_label]
series2 = df_returns[stock2_label]
plt.figure(1, figsize=(20,8))
plt.plot(serie)
plt.plot(series1_values)
plt.show()

NameError: name 'series2_values' is not defined

<Figure size 2000x800 with 0 Axes>

In [40]:
entry=2.0
exit=0.5
lookback=6
capital=10000
max_holding=6      # months
stop_loss_pct=0.10
df = pd.DataFrame({'p1': stock1_returns, 'p2': stock2_returns}).dropna().copy()
n = len(df)
if n < lookback + 1:
    raise ValueError("Not enough data for lookback")

trades = []
position = 0              # -1 short spread, +1 long spread, 0 flat
entry_spread = None
entry_date = None
entry_sigma = None
entry_index = None
holding = 0

# Pre-allocate columns
df['z'] = np.nan
df['action'] = ''
df['position'] = 0
df['pnl_unrealized'] = 0.0

for t in range(lookback, n):
    # Use history up to t-1 (no lookahead)
    hist = df.iloc[:t]
    y = hist['p1']
    x = hist['p2']
    X = sm.add_constant(x)
    print(f"X shape: {X.shape}")
    print(f"X columns: {X.columns}")
    print(f"X head:\n{X.head()}")
    model = sm.OLS(y, X).fit()
    print(f"Model params: {model.params}")
    beta = model.params[1]

X shape: (6, 1)
X columns: Index(['p2'], dtype='object')
X head:
               p2
date             
20140107  0.23195
20140114  0.23195
20140120  0.23195
20140122  0.23195
20140123  0.23195
Model params: p2   -0.027454
dtype: float64


IndexError: index 1 is out of bounds for axis 0 with size 1